In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
import pickle

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/nyc-flight/rewritten/o4_mini_high/checkpoints/post_cell_13_try_5.pickle

trying: ['df', 'flights_df']


me:  24
me:  24
trying: ['df', 'flights_df']
me:  24
me:  24
trying: ['names_for_count_']
me:  21
trying: ['np']
me:  0
trying: ['pd']
me:  0
trying: ['ds']
me:  22
trying: ['df2']
me:  17
trying: ['f']
me:  15
trying: ['orig_output']
me:  20
trying: ['f_total']
me:  15
trying: ['IPython']
me:  0
trying: ['plt']
me:  0
trying: ['sp']
me:  0
trying: ['matplotlib']
me:  0
trying: ['sklearn']
me:  0
trying: ['result']
me:  24
trying: ['time']
me:  0
trying: ['dt']
me:  23


Declaring variable np
Declaring variable pd
Declaring variable IPython
Declaring variable plt
Declaring variable sp
Declaring variable matplotlib
Declaring variable sklearn
Declaring variable time
Declaring variable f
Declaring variable f_total
Declaring variable df2
Declaring variable orig_output
Declaring variable names_for_count_
Declaring variable ds
Declaring variable dt
Declaring variable df
Declaring variable flights_df
Declaring variable df
Declaring variable flights_df
Declaring variable result


In [4]:
%%RecordEvent
%%time
### cell 14 ###

### cell 14 – optimized for cudf

# 1) group and count on GPU
# 2) filter on GPU (only the needed rows and columns)
# 3) build the message Series on GPU
# 4) attach it as a column, transfer once via Arrow, then print

df_grouped = (
    flights_df
      .groupby(['carrier', 'flight', 'dest'])
      .size()
      .reset_index(name='Size')
)

filtered = df_grouped[df_grouped['Size'] == 365][['carrier', 'flight', 'dest']]

msgs = (
    "Carrier: " + filtered['carrier']
    + ", Flight: " + filtered['flight'].astype(str)
    + ", Destination: " + filtered['dest']
)

# Assign the GPU‐built messages back into the DataFrame
filtered = filtered.assign(msg=msgs)

# One GPU→CPU transfer via Arrow on the DataFrame (not Series)
# then extract the column and convert to a Python list
msgs_list = filtered[['msg']].to_arrow().column(0).to_pylist()

# Print all lines at once
print("\n".join(msgs_list))

AttributeError: 'DataFrame' object has no attribute 'to_arrow'

In [ ]:
%Checkpoint /home/dias-benchmarks/new_notebooks/nyc-flight/rewritten/o4_mini_high/checkpoints/post_cell_14_try_8.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:
with open(
    "/home/dias-benchmarks/new_notebooks/nyc-flight/opt_cell_exec_info_14_try_8.pkl",
    "wb",
) as f:
    pickle.dump(opt_cell_exec_info[14], f)

In [ ]:
opt_output = Out.get(4)